# Revolutionizing Data Analysis with PandasGUI

Requires a real desktop display and native Qt/WebEngine; not Colab or a headless import-only test. This notebook opens an isolated sample-data window, operates its controls and closes it. No user data or clipboard is imported. Only code generated here from known operations is executed; never execute untrusted query expressions or generated code.

```bash
python -m pip install pandasgui==0.2.15 pandas==2.2.3 numpy==1.26.4 plotly==5.24.1 PyQt5==5.15.11 PyQtWebEngine==5.15.7 setuptools==80.9.0 ipython==8.37.0 wordcloud==1.9.6 matplotlib==3.11.2
```

Pinned pandas/Plotly/IPython/setuptools preserve the current PandasGUI integration APIs. Upstream warnings remain visible.

In [ ]:
import os, sys, tempfile, hashlib, urllib.request, io
from pathlib import Path
from importlib.metadata import version
from unittest.mock import patch
import numpy as np
import pandas as pd
# Isolate PandasGUI's sample-data/settings files for this demonstration.
work = tempfile.TemporaryDirectory(prefix='tdb-pandasgui-')
with patch('appdirs.user_data_dir', return_value=work.name):
    import pandasgui
from pandasgui import show
# Override the library's hardcoded Chromium log suppression before creating views.
# Software rendering avoids host-driver dependence; sandboxing stays enabled.
os.environ['QTWEBENGINE_CHROMIUM_FLAGS'] = '--disable-gpu --use-gl=swiftshader --enable-logging --log-level=0'
from PyQt5 import QtWidgets, QtCore, QtTest
from PyQt5.QtCore import Qt
print('Python:', sys.version.split()[0], 'PandasGUI:', version('pandasgui'), 'pandas:', pd.__version__)
print('No empty APPDATA workaround is applied.')

In [ ]:
REVISION = '2fcbd1fab0d107487b5c18cb86419166ffc99f43'
DATASETS = {
    'titanic': '81787d320d7f7b03df935e91de8bd19e11d45c5bbcab86ef4d4a76dc91b7d4f2',
    'iris': '9cc1c345c71bcc9b486b74cbf6063fa66f4bb5e0f603a4b3c3471ec2e5e8e355',
}
def load_sample(name):
    url = f'https://raw.githubusercontent.com/adamerose/datasets/{REVISION}/{name}.csv'
    with urllib.request.urlopen(url, timeout=30) as response:
        raw = response.read()
    if hashlib.sha256(raw).hexdigest() != DATASETS[name]:
        raise RuntimeError('Unexpected sample dataset bytes')
    return pd.read_csv(io.BytesIO(raw))
titanic, iris = load_sample('titanic'), load_sample('iris')
print('Titanic:', titanic.shape, 'Iris:', iris.shape)
print('Titanic filter columns:', titanic[['sex', 'pclass', 'survived', 'age']].dtypes.to_dict())
print('Iris columns:', ', '.join(iris.columns))

In [ ]:
gui = show(titanic=titanic, iris=iris, settings={'block': False, 'auto_finish': False})
app = gui.app
app.processEvents()
assert gui.isVisible() and app.platformName() not in ['offscreen', 'minimal']
pg = gui.store.data['titanic']
explorer = pg.dataframe_explorer
gui.resize(1100, 720)
app.processEvents()
print('Native Qt platform:', app.platformName(), 'Visible window:', gui.isVisible())
print('Named datasets:', ', '.join(gui.store.data))

In [ ]:
expressions = ['sex == "male"', 'pclass == 3', 'survived == 1', '30 < age < 40']
for expression in expressions:
    editor = explorer.filter_viewer.text_input
    editor.setText(expression)
    QtTest.QTest.keyClick(editor, Qt.Key_Return)
    app.processEvents()
expected = titanic.query(' and '.join(f'({e})' for e in expressions), engine='python')
pd.testing.assert_frame_equal(pg.df, expected)
assert len(pg.filters) == 4 and not any(f.failed for f in pg.filters)
print('Four filters:', expressions)
print('Matching passengers:', len(pg.df))
print(pg.df[['sex', 'pclass', 'survived', 'age']].to_string(index=False))

In [ ]:
model = explorer.filter_viewer.list_model
model.setData(model.index(0, 0), Qt.Unchecked, Qt.CheckStateRole)
app.processEvents()
assert not pg.filters[0].enabled
without_sex = titanic.query(' and '.join(f'({e})' for e in expressions[1:]), engine='python')
pd.testing.assert_frame_equal(pg.df, without_sex)
model.setData(model.index(0, 0), Qt.Checked, Qt.CheckStateRole)
assert pg.filters[0].enabled
# Sort through the actual data-view header's click handler.
header = explorer.dataframe_viewer.columnHeader
age_column = list(pg.df.columns).index('age')
header.scrollTo(header.model().index(0, age_column))
app.processEvents()
QtTest.QTest.mouseDClick(header.viewport(), Qt.LeftButton,
                       pos=header.visualRect(header.model().index(0, age_column)).center())
app.processEvents()
assert pg.sorted_column_name == 'age' and pg.df.age.is_monotonic_increasing
pg.refresh_statistics(force=True)
np.testing.assert_allclose(pg.column_statistics.loc['age', ['Mean', 'StdDev', 'Min', 'Max']].astype(float),
                           expected.age.agg(['mean', 'std', 'min', 'max']).to_numpy())
assert pg.column_statistics.loc['age', 'Count'] == expected.age.count()
print('Without sex filter:', len(without_sex), 'Restored:', len(pg.df))
print('Sorted column/state:', pg.sorted_column_name, pg.sort_state)
print(pg.column_statistics.loc[['age'], ['Count', 'Mean', 'StdDev', 'Min', 'Max']].to_string())

In [ ]:
# Remove all Titanic filters before reshaping.
while pg.filters:
    pg.remove_filter(0)
pg.sort_column(list(pg.df.columns).index('age'), 'None')
pd.testing.assert_frame_equal(pg.df, titanic)
reshaper = explorer.reshaper
# PandasGUI 0.2.15's Melt function names its first argument df, while the GUI
# dispatches data_frame=... . Adapt that keyword in this process only.
from pandasgui.widgets.reshaper import schemas as reshape_schemas
from pandasgui.jotly import ColumnNameList
from pandasgui.widgets.func_ui import Schema
def melt_compat(data_frame: pd.DataFrame, id_vars: ColumnNameList = None, value_vars: ColumnNameList = None):
    result = data_frame.melt(id_vars=id_vars, value_vars=value_vars)
    return result
melt_schema = next(schema for schema in reshape_schemas if schema.name == 'melt')
melt_schema.function = melt_compat
melt_schema.args = Schema(function=melt_compat).args
def reshape(kind, parameters):
    names = [reshaper.type_picker.item(i).text().lower() for i in range(reshaper.type_picker.count())]
    reshaper.type_picker.setCurrentRow(names.index(kind))
    app.processEvents()
    reshaper.func_ui.set_data(dict(parameters))
    QtTest.QTest.mouseClick(reshaper.func_ui.finish_button, Qt.LeftButton)
    app.processEvents()
    if reshaper.error_console.toPlainText():
        raise RuntimeError(reshaper.error_console.toPlainText())
    return reshaper.result
pivot = reshape('pivot', {'index': 'pclass', 'columns': 'sex', 'values': 'survived', 'aggfunc': 'mean'})
expected_pivot = titanic.pivot_table(index='pclass', columns='sex', values='survived', aggfunc='mean')
pd.testing.assert_frame_equal(pivot, expected_pivot)
melted = reshape('melt', {'id_vars': ['pclass'], 'value_vars': ['age', 'fare']})
expected_melt = titanic.melt(id_vars=['pclass'], value_vars=['age', 'fare'])
pd.testing.assert_frame_equal(melted, expected_melt)
print('Pivot: survival fraction by class and sex')
print(pivot.to_string())
print('Melt shape:', melted.shape)

In [ ]:
grapher = explorer.grapher
def plot(kind, parameters, controller=grapher):
    from pandasgui.widgets.grapher import schemas
    names = [schema.name for schema in schemas]
    controller.type_picker.setCurrentRow(names.index(kind))
    app.processEvents()
    controller.func_ui.set_data(dict(parameters))
    QtTest.QTest.mouseClick(controller.func_ui.finish_button, Qt.LeftButton)
    app.processEvents()
    if controller.error_console.toPlainText():
        raise RuntimeError(controller.error_console.toPlainText())
    return controller.fig
pie = plot('pie', {'names': 'sex'})
# Avoid the upstream mean aggregation over unrelated text columns.
fare_by_class = titanic.groupby('pclass', as_index=False).agg(fare=('fare', 'mean'))
gui.store.add_dataframe(fare_by_class, 'fare_by_class')
bar_controller = gui.store.data['fare_by_class'].dataframe_explorer.grapher
bar = plot('bar', {'x': 'pclass', 'y': ['fare'], 'aggregation': None, 'sort': None}, controller=bar_controller)
np.testing.assert_allclose(bar.data[0].y, fare_by_class.fare)
cloud = plot('word_cloud', {'words': 'embark_town'})
assert len(pie.data) and len(bar.data) and cloud is not None
print('Pie traces:', len(pie.data), 'Bar traces:', len(bar.data), 'Word cloud type:', type(cloud).__name__)

In [ ]:
iris_pg = gui.store.data['iris']
iris_pg.add_filter('sepal_length > 7')
pd.testing.assert_frame_equal(iris_pg.df, iris.query('sepal_length > 7', engine='python'))
iris_pg.sort_column(list(iris.columns).index('sepal_width'), 'Desc')
assert iris_pg.df.sepal_width.is_monotonic_decreasing
iris_pg.change_column_type(list(iris.columns).index('species'), 'category')
assert str(iris_pg.df.species.dtype) == 'category'
# Fix this known upstream history-string defect before exporting/replaying it:
# the generated dtype-change line omits quotes around column and dtype names.
dtype_history = next(item for item in reversed(iris_pg.history) if item.comment == 'change_column_type')
dtype_history.code = "df['species'] = df['species'].astype('category')"
iris_pg.refresh_statistics(force=True)
np.testing.assert_allclose(iris_pg.column_statistics.loc['sepal_length', 'Mean'], iris_pg.df.sepal_length.mean())
iris_view = iris_pg.dataframe_explorer.dataframe_viewer
iris_view.color_mode = 'column'
iris_model = iris_view.dataView.model()
assert iris_model.data(iris_model.index(0, 0), Qt.BackgroundRole) is not None
print('Iris filtered rows:', len(iris_pg.df), 'Species dtype:', iris_pg.df.species.dtype)
# Use the actual Iris Grapher controller and Finish signal for a 3D scatter plot.
iris_grapher = iris_pg.dataframe_explorer.grapher
from pandasgui.widgets.grapher import schemas
iris_grapher.type_picker.setCurrentRow([s.name for s in schemas].index('scatter_3d'))
app.processEvents()
iris_grapher.func_ui.set_data({'x': 'sepal_length', 'y': 'sepal_width', 'z': 'petal_length', 'color': 'species'})
QtTest.QTest.mouseClick(iris_grapher.func_ui.finish_button, Qt.LeftButton)
app.processEvents()
if iris_grapher.error_console.toPlainText():
    raise RuntimeError(iris_grapher.error_console.toPlainText())
assert all(trace.type == 'scatter3d' for trace in iris_grapher.fig.data)
print('3D scatter traces:', len(iris_grapher.fig.data))
iris_reshaper = iris_pg.dataframe_explorer.reshaper
iris_reshaper.type_picker.setCurrentRow([iris_reshaper.type_picker.item(i).text().lower() for i in range(iris_reshaper.type_picker.count())].index('pivot'))
app.processEvents()
iris_reshaper.func_ui.set_data({'columns': 'species', 'values': 'petal_length', 'aggfunc': 'mean'})
QtTest.QTest.mouseClick(iris_reshaper.func_ui.finish_button, Qt.LeftButton)
app.processEvents()
if iris_reshaper.error_console.toPlainText():
    raise RuntimeError(iris_reshaper.error_console.toPlainText())
pd.testing.assert_frame_equal(iris_reshaper.result, pd.pivot_table(iris_pg.df, columns='species', values='petal_length', aggfunc='mean', observed=False))
print('Iris pivot matches independent group means.')

In [ ]:
import ast
exported_history = iris_pg.code_export()
namespace = {'df': iris.copy()}
exec(compile(exported_history, '<PandasGUI generated history>', 'exec'), namespace)
pd.testing.assert_frame_equal(namespace['df'], iris_pg.df)
print(exported_history)
# The plot/reshape Code Export dialogs expose the same generated code used by the GUI.
iris_grapher.func_ui.code_export()
plot_code = iris_grapher.func_ui.code_export_dialog_text.toPlainText()
reshaper.func_ui.code_export()
reshape_code = reshaper.func_ui.code_export_dialog_text.toPlainText()
# GUI exports are extracted function bodies. Convert only top-level return/show
# statements to a result assignment so the known generated snippets run as scripts.
def runnable_export(source):
    tree = ast.parse(source)
    for i, node in enumerate(tree.body):
        if isinstance(node, ast.Return):
            tree.body[i] = ast.Assign(targets=[ast.Name(id='result', ctx=ast.Store())], value=node.value)
        elif isinstance(node, ast.Expr) and isinstance(node.value, ast.Call) and isinstance(node.value.func, ast.Name) and node.value.func.id == 'show':
            tree.body[i] = ast.Assign(targets=[ast.Name(id='result', ctx=ast.Store())], value=node.value.args[0])
    return ast.fix_missing_locations(tree)
plot_namespace = {'df': iris_pg.df.copy()}
exec(compile(runnable_export(plot_code), '<PandasGUI plot export>', 'exec'), plot_namespace)
reshape_namespace = {'df': titanic.copy()}
exec(compile(runnable_export(reshape_code), '<PandasGUI reshape export>', 'exec'), reshape_namespace)
pd.testing.assert_frame_equal(reshape_namespace['result'], expected_melt)
assert plot_namespace['result'] is not None
print('Generated history and normalized plot/reshape snippets execute; dataframe results match independently.')

In [ ]:
# Confirm that the real embedded browser loaded the Plotly plot, not only its Python object.
view = iris_grapher.figure_viewer
rendered = []
for attempt in range(50):
    view.page().runJavaScript("Boolean(document.querySelector('.js-plotly-plot') && document.querySelector('.js-plotly-plot').data)", rendered.append)
    QtTest.QTest.qWait(100)
    if rendered and rendered[-1]:
        break
assert rendered and rendered[-1], 'The embedded Plotly page did not render'
gui.navigator.setCurrentItem(gui.navigator.topLevelItem(0))
gui.resize(1100, 640)
app.processEvents()
gui.grab().save(str(Path(work.name) / 'native-window.png'))
if os.getenv('TDB_GUI_SCREENSHOT'):
    gui.grab().save(os.environ['TDB_GUI_SCREENSHOT'])
assert gui.isVisible()
print('PASS: native visible GUI, real input/header/Finish callbacks, independent filters/statistics/pivot/melt, plots, dtype/sort, generated history and embedded Plotly rendering.')
gui.close()
app.processEvents()
QtTest.QTest.qWait(300)
assert not gui.isVisible()
work.cleanup()
print('Demonstration window closed; isolated profile removed.')

## Original source-code appendix

Original five blocks are retained verbatim as nonexecuting source. Use the maintained setup above, not the old empty APPDATA workaround.

```text
pip install pandasgui
```

```text
import pandas as pd
import pandasgui
```

```text
import os
os.environ['APPDATA'] = ""
```

```text
from pandasgui.datasets import titanic
```

```text
pandasgui.show(titanic)
```